In [ ]:
import os
import torch
import relbench

/home/xuewenqi/relbench/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import numpy as np

from torch.nn import BCEWithLogitsLoss, L1Loss
from relbench.datasets import get_dataset
from relbench.tasks import get_task

dataset = get_dataset("rel-f1", download=True)
task = get_task("rel-f1", "driver-position", download=True)

train_table = task.get_table("train")
val_table = task.get_table("val")
test_table = task.get_table("test")

out_channels = 1
loss_fn = L1Loss()
tune_metric = "mae"
higher_is_better = False

Let's check out the training table just to make sure it looks fine.

In [ ]:
train_table

Table(df=
           date  driverId  position
0    2004-07-05        10     10.75
1    2004-07-05        47     12.00
2    2004-03-07         7     15.00
3    2004-01-07        10      9.00
4    2003-09-09        52     13.00
...         ...       ...       ...
7448 1995-08-22        96     15.75
7449 1975-06-08       228      8.00
7450 1965-05-31       418     16.00
7451 1961-08-20       467     37.00
7452 1954-05-29       677     30.00

[7453 rows x 3 columns],
  fkey_col_to_pkey_table={'driverId': 'drivers'},
  pkey_col=None,
  time_col=date)

Note that to load the data we did not require any deep learning libraries. Now we introduce the PyTorch Frame library, which is useful for encoding individual tables into initial node features.

In [ ]:
import os
import math
import numpy as np
from tqdm import tqdm

import torch
import torch_geometric
import torch_frame

# Some book keeping
from torch_geometric.seed import seed_everything

seed_everything(42)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)  # check that it's cuda if you want it to run in reasonable time!
root_dir = "./data"

cuda


The first big move is to build a graph out of the database. Here we use our pre-prepared conversion function.

The source code can be found at: https://github.com/snap-stanford/relbench/blob/main/relbench/modeling/graph.py

Each node in the graph corresonds to a single row in the database. Crucially, PyTorch Frame stores whole tables as objects in a way that is compatibile with PyG minibatch sampling, meaning we can sample subgraphs as in https://arxiv.org/abs/1706.02216, and retrieve the relevant raw features.

PyTorch Frame also stores the `stype` (i.e., modality) of each column, and any specialized feature encoders (e.g., text encoders) to be used later. So we need to configure the `stype` for each column, for which we use a function that tries to automatically detect the `stype`.

In [ ]:
from relbench.modeling.utils import get_stype_proposal

db = dataset.get_db()
col_to_stype_dict = get_stype_proposal(db)
col_to_stype_dict

Loading Database object from /home/xuewenqi/.cache/relbench/rel-f1/db...
Done in 0.06 seconds.


{'standings': {'driverStandingsId': <stype.numerical: 'numerical'>,
  'raceId': <stype.numerical: 'numerical'>,
  'driverId': <stype.numerical: 'numerical'>,
  'points': <stype.numerical: 'numerical'>,
  'position': <stype.numerical: 'numerical'>,
  'wins': <stype.numerical: 'numerical'>,
  'date': <stype.timestamp: 'timestamp'>},
 'constructors': {'constructorId': <stype.numerical: 'numerical'>,
  'constructorRef': <stype.text_embedded: 'text_embedded'>,
  'name': <stype.text_embedded: 'text_embedded'>,
  'nationality': <stype.text_embedded: 'text_embedded'>},
 'circuits': {'circuitId': <stype.numerical: 'numerical'>,
  'circuitRef': <stype.text_embedded: 'text_embedded'>,
  'name': <stype.text_embedded: 'text_embedded'>,
  'location': <stype.text_embedded: 'text_embedded'>,
  'country': <stype.text_embedded: 'text_embedded'>,
  'lat': <stype.numerical: 'numerical'>,
  'lng': <stype.numerical: 'numerical'>,
  'alt': <stype.numerical: 'numerical'>},
 'races': {'raceId': <stype.numerica

If trying a new dataset, you should definitely check through this dict of `stype`s to check that look right, and manually change any mistakes by the auto-detection function.

Next we also define our text encoding model, which we use GloVe embeddings for speed and convenience. Feel free to try alternatives here.

In [ ]:
from typing import List, Optional
from sentence_transformers import SentenceTransformer
from torch import Tensor


class GloveTextEmbedding:
    def __init__(self, device: Optional[torch.device] = None):
        self.model = SentenceTransformer(
            "sentence-transformers/average_word_embeddings_glove.6B.300d",
            device=device,
        )

    def __call__(self, sentences: List[str]) -> Tensor:
        return torch.from_numpy(self.model.encode(sentences))

In [ ]:
from torch_frame.config.text_embedder import TextEmbedderConfig
from relbench.modeling.graph import make_pkey_fkey_graph

text_embedder_cfg = TextEmbedderConfig(
    text_embedder=GloveTextEmbedding(device=device), batch_size=256
)

data, col_stats_dict = make_pkey_fkey_graph(
    db,
    col_to_stype_dict=col_to_stype_dict,  # speficied column types
    text_embedder_cfg=text_embedder_cfg,  # our chosen text encoder
    cache_dir=os.path.join(
        root_dir, f"rel-f1_materialized_cache"
    ),  # store materialized graph for convenience
)


/home/xuewenqi/relbench/.venv/lib/python3.10/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([scalar])` to allowlist this global.
  self._tensor_frame, self._col_stats = torch_frame.load(
/home/xuewenqi/relbench/.venv/lib/python3.10/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([scalar])` to allowlist this global.
  self._tensor_frame, self._col_stats = torch_frame.load(
/home/xuewenqi/relbench/.venv/lib/python3.10/site-packages/torch_frame/data/dataset.py:587: UserWarning: Weights only load failed. Please file an issue to make `torch.load(weights_only=True)` compatible in your case. Please use `torch.serialization.add_safe_globals([scalar])` t

In [ ]:
from torch_frame import TensorFrame 
tf: TensorFrame = data['circuits'].tf
tf.get_col_feat('circuitRef')
# 获取 circuitRef 列的特征 Tensor
feat = tf.get_col_feat('circuitRef')

# 直接打印整个 Tensor
print(feat)

# 打印 Tensor 的维度信息
print(f"Shape: {feat.shape}")
# 打印前 10 行数据
print(feat[:10])
# 打印该列所有嵌入向量的原始数值
print(tf.get_col_feat('circuitRef').values)

MultiEmbeddingTensor(num_rows=77, num_cols=1, device='cpu')
Shape: (77, 1, -1)
MultiEmbeddingTensor(num_rows=10, num_cols=1, device='cpu')
tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [-0.5573, -0.7211, -0.0695,  ..., -0.0442,  0.5663, -0.0110],
        [ 0.6028, -0.0052, -0.0579,  ..., -0.3377, -0.4982, -0.4881],
        ...,
        [ 0.1567, -0.1249,  0.0884,  ...,  0.1978, -0.7723, -0.3883],
        [ 0.1259, -0.5524,  0.5465,  ...,  0.5631,  0.6860,  0.0645],
        [ 0.3977,  0.0085, -0.0658,  ..., -0.4590, -0.1766,  0.2713]])


We can now check out `data`, our main graph object. `data` is a heterogeneous and temporal graph, with node types given by the table it originates from.

In [ ]:
data

HeteroData(
  standings={
    tf=TensorFrame([28115, 4]),
    time=[28115],
  },
  constructors={ tf=TensorFrame([211, 3]) },
  circuits={ tf=TensorFrame([77, 7]) },
  races={
    tf=TensorFrame([820, 5]),
    time=[820],
  },
  drivers={ tf=TensorFrame([857, 6]) },
  constructor_results={
    tf=TensorFrame([9408, 2]),
    time=[9408],
  },
  results={
    tf=TensorFrame([20323, 11]),
    time=[20323],
  },
  qualifying={
    tf=TensorFrame([4082, 3]),
    time=[4082],
  },
  constructor_standings={
    tf=TensorFrame([10170, 4]),
    time=[10170],
  },
  (standings, f2p_raceId, races)={ edge_index=[2, 28115] },
  (races, rev_f2p_raceId, standings)={ edge_index=[2, 28115] },
  (standings, f2p_driverId, drivers)={ edge_index=[2, 28115] },
  (drivers, rev_f2p_driverId, standings)={ edge_index=[2, 28115] },
  (races, f2p_circuitId, circuits)={ edge_index=[2, 820] },
  (circuits, rev_f2p_circuitId, races)={ edge_index=[2, 820] },
  (constructor_results, f2p_raceId, races)={ edge_index=[2,

We can also check out the TensorFrame for one table like this:

In [ ]:
data["races"].tf

TensorFrame(
  num_cols=5,
  num_rows=820,
  categorical (1): ['year'],
  numerical (1): ['round'],
  timestamp (2): ['date', 'time'],
  embedding (1): ['name'],
  has_target=False,
  device='cpu',
)

This may be a little confusing at first, as in graph ML it is more standard to associate to the graph object `data` a tensor, e.g., `data.x` for which `data.x[idx]` is a 1D array/tensor storing all the features for node with index `idx`.

But actually this `data` object behaves similarly. For a given node type, e.g., `races` again, `data['races']` stores two pieces of information


In [ ]:
list(data["races"].keys())

['tf', 'time']

A `TensorFrame` object, and a timestamp for each node. The `TensorFrame` object acts analogously to the usual tensor of node features, and you can simply use indexing to retrieve the features of a single row (node), or group of nodes.

In [ ]:
data["races"].tf[10]

TensorFrame(
  num_cols=5,
  num_rows=1,
  categorical (1): ['year'],
  numerical (1): ['round'],
  timestamp (2): ['date', 'time'],
  embedding (1): ['name'],
  has_target=False,
  device='cpu',
)

In [ ]:
data["races"].tf[10:20]

TensorFrame(
  num_cols=5,
  num_rows=10,
  categorical (1): ['year'],
  numerical (1): ['round'],
  timestamp (2): ['date', 'time'],
  embedding (1): ['name'],
  has_target=False,
  device='cpu',
)

We can also check the edge indices between two different node types, such as `races` amd `circuits`. Note that the edges are also heterogenous, so we also need to specify which edge type we want to look at. Here we look at `f2p_curcuitId`, which are the directed edges pointing _from_ a race (the `f` stands for `foreign key`), _to_ the circuit at which te race happened (the `p` stands for `primary key`).

In [ ]:
data[("races", "f2p_circuitId", "circuits")]

{'edge_index': tensor([[  0,   1,   2,  ..., 817, 818, 819],
        [  8,   5,  18,  ...,  21,  17,  23]])}

Now we are ready to instantiate our data loaders. For this we will need to import PyTorch Geometric, our GNN library. Whilst we're at it let's add a seed.


In [ ]:
from relbench.modeling.graph import get_node_train_table_input, make_pkey_fkey_graph
from torch_geometric.loader import NeighborLoader

loader_dict = {}

for split, table in [
    ("train", train_table),
    ("val", val_table),
    ("test", test_table),
]:
    table_input = get_node_train_table_input(
        table=table,
        task=task,
    )
    entity_table = table_input.nodes[0]
    loader_dict[split] = NeighborLoader(
        data,
        num_neighbors=[
            128 for i in range(2)
        ],  # we sample subgraphs of depth 2, 128 neighbors per node.
        time_attr="time",
        input_nodes=table_input.nodes,
        input_time=table_input.time,
        transform=table_input.transform,
        batch_size=512,
        temporal_strategy="uniform",
        shuffle=split == "train",
        num_workers=0,
        persistent_workers=False,
    )

Now we need our model...




In [ ]:
from torch.nn import BCEWithLogitsLoss
import copy
from typing import Any, Dict, List

import torch
from torch import Tensor
from torch.nn import Embedding, ModuleDict
from torch_frame.data.stats import StatType
from torch_geometric.data import HeteroData
from torch_geometric.nn import MLP
from torch_geometric.typing import NodeType

from relbench.modeling.nn import HeteroEncoder, HeteroGraphSAGE, HeteroTemporalEncoder


class Model(torch.nn.Module):

    def __init__(
        self,
        data: HeteroData,
        col_stats_dict: Dict[str, Dict[str, Dict[StatType, Any]]],
        num_layers: int,
        channels: int,
        out_channels: int,
        aggr: str,
        norm: str,
        # List of node types to add shallow embeddings to input
        shallow_list: List[NodeType] = [],
        # ID awareness
        id_awareness: bool = False,
    ):
        super().__init__()

        self.encoder = HeteroEncoder(
            channels=channels,
            node_to_col_names_dict={
                node_type: data[node_type].tf.col_names_dict
                for node_type in data.node_types
            },
            node_to_col_stats=col_stats_dict,
        )
        self.temporal_encoder = HeteroTemporalEncoder(
            node_types=[
                node_type for node_type in data.node_types if "time" in data[node_type]
            ],
            channels=channels,
        )
        self.gnn = HeteroGraphSAGE(
            node_types=data.node_types,
            edge_types=data.edge_types,
            channels=channels,
            aggr=aggr,
            num_layers=num_layers,
        )
        self.head = MLP(
            channels,
            out_channels=out_channels,
            norm=norm,
            num_layers=1,
        )
        self.embedding_dict = ModuleDict(
            {
                node: Embedding(data.num_nodes_dict[node], channels)
                for node in shallow_list
            }
        )

        self.id_awareness_emb = None
        if id_awareness:
            self.id_awareness_emb = torch.nn.Embedding(1, channels)
        self.reset_parameters()

    def reset_parameters(self):
        self.encoder.reset_parameters()
        self.temporal_encoder.reset_parameters()
        self.gnn.reset_parameters()
        self.head.reset_parameters()
        for embedding in self.embedding_dict.values():
            torch.nn.init.normal_(embedding.weight, std=0.1)
        if self.id_awareness_emb is not None:
            self.id_awareness_emb.reset_parameters()

    def forward(
        self,
        batch: HeteroData,
        entity_table: NodeType,
    ) -> Tensor:
        seed_time = batch[entity_table].seed_time
        x_dict = self.encoder(batch.tf_dict)

        rel_time_dict = self.temporal_encoder(
            seed_time, batch.time_dict, batch.batch_dict
        )

        for node_type, rel_time in rel_time_dict.items():
            x_dict[node_type] = x_dict[node_type] + rel_time

        for node_type, embedding in self.embedding_dict.items():
            x_dict[node_type] = x_dict[node_type] + embedding(batch[node_type].n_id)

        x_dict = self.gnn(
            x_dict,
            batch.edge_index_dict,
            batch.num_sampled_nodes_dict,
            batch.num_sampled_edges_dict,
        )

        return self.head(x_dict[entity_table][: seed_time.size(0)])

    def forward_dst_readout(
        self,
        batch: HeteroData,
        entity_table: NodeType,
        dst_table: NodeType,
    ) -> Tensor:
        if self.id_awareness_emb is None:
            raise RuntimeError(
                "id_awareness must be set True to use forward_dst_readout"
            )
        seed_time = batch[entity_table].seed_time
        x_dict = self.encoder(batch.tf_dict)
        # Add ID-awareness to the root node
        x_dict[entity_table][: seed_time.size(0)] += self.id_awareness_emb.weight

        rel_time_dict = self.temporal_encoder(
            seed_time, batch.time_dict, batch.batch_dict
        )

        for node_type, rel_time in rel_time_dict.items():
            x_dict[node_type] = x_dict[node_type] + rel_time

        for node_type, embedding in self.embedding_dict.items():
            x_dict[node_type] = x_dict[node_type] + embedding(batch[node_type].n_id)

        x_dict = self.gnn(
            x_dict,
            batch.edge_index_dict,
        )

        return self.head(x_dict[dst_table])


model = Model(
    data=data,
    col_stats_dict=col_stats_dict,
    num_layers=2,
    channels=128,
    out_channels=1,
    aggr="sum",
    norm="batch_norm",
).to(device)


# if you try out different RelBench tasks you will need to change these
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
epochs = 10

We also need standard train/test loops

In [ ]:
def train() -> float:
    model.train()

    loss_accum = count_accum = 0
    for batch in tqdm(loader_dict["train"]):
        batch = batch.to(device)

        optimizer.zero_grad()
        pred = model(
            batch,
            task.entity_table,
        )
        pred = pred.view(-1) if pred.size(1) == 1 else pred

        loss = loss_fn(pred.float(), batch[entity_table].y.float())
        loss.backward()
        optimizer.step()

        loss_accum += loss.detach().item() * pred.size(0)
        count_accum += pred.size(0)

    return loss_accum / count_accum


@torch.no_grad()
def test(loader: NeighborLoader) -> np.ndarray:
    model.eval()

    pred_list = []
    for batch in loader:
        batch = batch.to(device)
        pred = model(
            batch,
            task.entity_table,
        )
        pred = pred.view(-1) if pred.size(1) == 1 else pred
        pred_list.append(pred.detach().cpu())
    return torch.cat(pred_list, dim=0).numpy()

Now we are ready to train!

In [ ]:
state_dict = None
best_val_metric = -math.inf if higher_is_better else math.inf
for epoch in range(1, epochs + 1):
    train_loss = train()
    val_pred = test(loader_dict["val"])
    val_metrics = task.evaluate(val_pred, val_table)
    print(f"Epoch: {epoch:02d}, Train loss: {train_loss}, Val metrics: {val_metrics}")

    if (higher_is_better and val_metrics[tune_metric] > best_val_metric) or (
        not higher_is_better and val_metrics[tune_metric] < best_val_metric
    ):
        best_val_metric = val_metrics[tune_metric]
        state_dict = copy.deepcopy(model.state_dict())


model.load_state_dict(state_dict)
val_pred = test(loader_dict["val"])
val_metrics = task.evaluate(val_pred, val_table)
print(f"Best Val metrics: {val_metrics}")

test_pred = test(loader_dict["test"])
test_metrics = task.evaluate(test_pred)
print(f"Best test metrics: {test_metrics}")

  0%|          | 0/15 [00:00<?, ?it/s]

  0%|          | 0/15 [00:02<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 12.00 MiB. GPU 0 has a total capacity of 23.69 GiB of which 13.94 MiB is free. Process 3499863 has 254.00 MiB memory in use. Process 139187 has 1.19 GiB memory in use. Process 444221 has 1.07 GiB memory in use. Process 457860 has 254.00 MiB memory in use. Process 2386674 has 6.55 GiB memory in use. Process 3239140 has 3.39 GiB memory in use. Process 3512186 has 254.00 MiB memory in use. Process 1168741 has 254.00 MiB memory in use. Process 1175545 has 254.00 MiB memory in use. Process 1266672 has 254.00 MiB memory in use. Process 1298677 has 254.00 MiB memory in use. Process 1358253 has 1.50 GiB memory in use. Process 1665082 has 254.00 MiB memory in use. Process 1669301 has 1.59 GiB memory in use. Process 1673520 has 1.21 GiB memory in use. Process 1676929 has 1.01 GiB memory in use. Process 1678667 has 1.05 GiB memory in use. Including non-PyTorch memory, this process has 1.52 GiB memory in use. Process 1698496 has 1.55 GiB memo

In [ ]:
import os
import copy
import math
import torch
import numpy as np
import optuna
import logging
from tqdm import tqdm

# ==========================================
# 1. 配置详细的日志记录 (输出到控制台和文件)
# ==========================================
log_file = "optuna_tuning.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    handlers=[
        logging.FileHandler(log_file),      # 保存到文件用于复盘
        logging.StreamHandler()             # 输出到控制台看实时进度
    ]
)
logger = logging.getLogger(__name__)

# ==========================================
# 2. 改造基础函数 (避免全局变量污染 Trial)
# ==========================================
def train(model, optimizer, loader, device, task, loss_fn, entity_table) -> float:
    model.train()
    loss_accum = count_accum = 0
    
    # 调参时如果觉得 tqdm 刷屏，可将其 disable=True
    for batch in tqdm(loader, desc="Training", leave=False):
        batch = batch.to(device)

        optimizer.zero_grad()
        pred = model(
            batch,
            task.entity_table,
        )
        pred = pred.view(-1) if pred.size(1) == 1 else pred

        loss = loss_fn(pred.float(), batch[entity_table].y.float())
        loss.backward()
        optimizer.step()

        loss_accum += loss.detach().item() * pred.size(0)
        count_accum += pred.size(0)

    return loss_accum / count_accum


@torch.no_grad()
def test(model, loader, device, task) -> np.ndarray:
    model.eval()
    pred_list = []
    
    for batch in loader:
        batch = batch.to(device)
        pred = model(
            batch,
            task.entity_table,
        )
        pred = pred.view(-1) if pred.size(1) == 1 else pred
        pred_list.append(pred.detach().cpu())
        
    return torch.cat(pred_list, dim=0).numpy()


# ==========================================
# 3. 定义 Optuna Objective (目标函数)
# ==========================================
def objective(trial):
    logger.info(f"--- 开始 Trial {trial.number} ---")
    
    # [超参数空间定义与采样]
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    num_layers = trial.suggest_int("num_layers", 2, 4)
    channels = trial.suggest_categorical("channels", [64, 128, 256])
    aggr = trial.suggest_categorical("aggr", ["sum", "mean", "max"])
    
    logger.info(f"参数采样: lr={lr:.5f}, num_layers={num_layers}, channels={channels}, aggr={aggr}")

    # [模型定义]
    # 注意: data, col_stats_dict, device 等需要作为全局变量或通过闭包传入
    model = Model(
        data=data,
        col_stats_dict=col_stats_dict,
        num_layers=num_layers,
        channels=channels,
        out_channels=1,
        aggr=aggr,
        norm="batch_norm",
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # [初始化指标]
    epochs = 10
    state_dict = None
    best_val_metric = -math.inf if higher_is_better else math.inf
    
    # [训练循环]
    for epoch in range(1, epochs + 1):
        # 1. 训练与验证
        train_loss = train(model, optimizer, loader_dict["train"], device, task, loss_fn, entity_table)
        val_pred = test(model, loader_dict["val"], device, task)
        val_metrics = task.evaluate(val_pred, val_table)
        
        current_metric = val_metrics[tune_metric]
        
        # 2. 打印详细日志
        logger.info(
            f"Trial {trial.number:03d} | Epoch: {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val {tune_metric}: {current_metric:.4f} | "
            f"Full Val Metrics: {val_metrics}"
        )

        # 3. 保存当前 Trial 的最佳模型
        is_best = False
        if higher_is_better and current_metric > best_val_metric:
            is_best = True
        elif not higher_is_better and current_metric < best_val_metric:
            is_best = True

        if is_best:
            best_val_metric = current_metric
            state_dict = copy.deepcopy(model.state_dict())
            logger.info(f"--> [Trial {trial.number}] 找到目前最好 {tune_metric}: {best_val_metric:.4f}")

        # 4. Optuna 剪枝机制 (报告中间结果，若极差则提前终止)
        # 注意: 若 higher_is_better=False (如 RMSE)，直接传入；若是 AUC 等，可能需要调换符号或设置 study 方向。
        trial.report(current_metric, epoch)
        if trial.should_prune():
            logger.warning(f"Trial {trial.number} 在 Epoch {epoch} 被剪枝 (Pruned).")
            raise optuna.exceptions.TrialPruned()

    # [测试集评估阶段] - 仅在这个 Trial 顺利完成且找到最好权重时执行
    if state_dict is not None:
        model.load_state_dict(state_dict)
        test_pred = test(model, loader_dict["test"], device, task)
        test_metrics = task.evaluate(test_pred)
        logger.info(f"Trial {trial.number} 最终 Test metrics: {test_metrics}")
        
        # 将 Test 结果存在 user_attrs 中，方便事后分析
        trial.set_user_attr("test_metrics", test_metrics)

    logger.info(f"--- 结束 Trial {trial.number}，最佳 Val {tune_metric}: {best_val_metric:.4f} --- \n")
    return best_val_metric


# ==========================================
# 4. 启动调参 Study
# ==========================================
if __name__ == "__main__":
    # 假设你前面的数据加载和环境变量已经准备好...
    # data, loader_dict, device, task, loss_fn, entity_table, val_table, tune_metric, higher_is_better = ...

    # 确定优化的方向
    direction = "maximize" if higher_is_better else "minimize"
    
    logger.info(f"初始化 Optuna Study，优化方向: {direction} (依据 {tune_metric})")
    
    study = optuna.create_study(
        study_name="relbench-gnn-study",
        direction=direction,
        pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=2) # 前3个trial不剪枝，前2个epoch不剪枝
    )

    # 运行搜索，捕获键盘中断以方便随时终止
    try:
        study.optimize(objective, n_trials=20)
    except KeyboardInterrupt:
        logger.warning("接收到停止信号，提前终止搜索。")

    # ==========================================
    # 5. 打印最终的最优结果
    # ==========================================
    logger.info("=" * 50)
    logger.info("超参数搜索完成！")
    logger.info(f"最佳 Trial 编号: {study.best_trial.number}")
    logger.info(f"最佳 Val {tune_metric}: {study.best_trial.value}")
    logger.info("最佳参数组合:")
    for key, value in study.best_trial.params.items():
        logger.info(f"    {key}: {value}")
        
    if "test_metrics" in study.best_trial.user_attrs:
        logger.info(f"该参数下的 Test Metrics: {study.best_trial.user_attrs['test_metrics']}")
    logger.info(f"所有训练细节已保存至 {log_file}，你可以通过搜索该文件进行分析。")